In [ ]:
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import numpy as np
import spacy

In [ ]:
# 1. Chargement optimisé (on désactive ce qui est lourd et inutile pour la lemmatisation)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def lemmatize_word(word: str) -> str:
    doc = nlp(word)
    
    lemmas = []
    for token in doc:
        if token.text.lower() == "not" or token.lemma_ == "-PRON-":
            lemmas.append(token.text.lower())
        else:
            lemmas.append(token.lemma_.lower())
            
    return "_".join(lemmas)

# --- Test rapide ---
print(lemmatize_word("running dogs")) # -> run_dog
print(lemmatize_word("not sleeping")) # -> not_sleep

In [ ]:
bi_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device = "cuda:1")
cross_model = CrossEncoder('cross-encoder/stsb-roberta-base', device = "cuda:1")
cross_model_2 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device = "cuda:1")

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(scores):
    exp_x = np.exp(scores)
    return exp_x/np.sum(exp_x)

def linear_meanmax(scores, alpha = 0.5):
    return alpha*np.mean(scores)+(1-alpha)*np.max(scores)

def log_sum_exp_pooling(scores, r=5.0):
    k = len(scores)
    max_s = np.max(scores)
    sum_exp = sum(np.exp(r * (s - max_s)) for s in scores)
    return max_s + (1/r) * np.log(sum_exp / k)

def similarity_score(sentence_a, sentence_b):
    bi_embeddings = bi_model.encode([sentence_a, sentence_b])
    bi_score =  util.cos_sim(bi_embeddings[0], bi_embeddings[1])
    cross_score_stsb = cross_model.predict((sentence_a, sentence_b))
    #cross_score_marco = cross_model_2.predict((sentence_a, sentence_b))
    res = {
        "bi-encoder" : bi_score,
        "cross-encoder/stsb-roberta-base" : cross_score_stsb,
        #"cross-encoder/ms-marco-MiniLM-L-6-v2" : cross_score_marco,
        "linear_meanmax" : linear_meanmax([bi_score.item(), cross_score_stsb]),
        "log_sum_exp_pooling" : log_sum_exp_pooling([bi_score.item(), cross_score_stsb])
    }
    return res

## test scores

In [ ]:
similarity_score("2004", "in 2004")


In [ ]:
similarity_score("tourism", "travel")

In [ ]:
similarity_score("tourism", "tourist")

il faut comparer le sens en fonction de la classe grammaticale + contexte (?)

In [ ]:
similarity_score("tourists", "tourist")

In [ ]:
similarity_score("2004", "15005")


In [ ]:
similarity_score("eat", "eating")

In [ ]:
similarity_score("new york", "new york city")

In [ ]:
similarity_score("united states", "white house")


In [ ]:
similarity_score("united states", "US")


In [ ]:
similarity_score("eat", "drink")


In [ ]:
similarity_score("eat", "see")


In [ ]:
similarity_score("eat", "food")


In [ ]:
similarity_score("eat", "not eat")


In [ ]:
similarity_score("boy", "girl")


In [ ]:
similarity_score("intern", "interns")

In [ ]:
similarity_score("moroccan", "morocco")

In [ ]:
similarity_score("John is eating ham.", "John is eating chicken.")


In [ ]:
similarity_score("chicken isA ham.", "John is eating ham.")


In [ ]:
similarity_score("eating relatedTo ham.", "John is eating ham.")


## pooling test

In [ ]:
s1 = np.array([0.9,0.9,0.9])
s2 = np.array([0.2,0.9])
s3 = np.array([0.3,0.6,0.9])

In [ ]:
linear_meanmax(s1), linear_meanmax(s2), linear_meanmax(s3)

In [ ]:
log_sum_exp_pooling(s1), log_sum_exp_pooling(s2), log_sum_exp_pooling(s3)


In [ ]:
z(s1), z(s2), z(s3)


## -